In [1]:
from nuscenes.nuscenes import NuScenes

# Initialize the dataset using the relative path you just created
nusc = NuScenes(version='v1.0-mini', dataroot='./data/sets/nuscenes', verbose=True)

# Print out the available scenes to confirm success
nusc.list_scenes()
from nuscenes.utils.data_classes import RadarPointCloud
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from nuscenes.utils.data_classes import RadarPointCloud, LidarPointCloud
from nuscenes.map_expansion.map_api import NuScenesMap
from pyquaternion import Quaternion


Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.400 seconds.
Reverse indexing ...
Done reverse indexing in 0.0 seconds.
scene-0061, Parked truck, construction, intersectio... [18-07-24 03:28:47]   19s, singapore-onenorth, #anns:4622
scene-0103, Many peds right, wait for turning car, ... [18-08-01 19:26:43]   19s, boston-seaport, #anns:2046
scene-0655, Parking lot, parked cars, jaywalker, be... [18-08-27 15:51:32]   20s, boston-seaport, #anns:2332
scene-0553, Wait at intersection, bicycle, large tr... [18-08-28 20:48:16]   20s, boston-seaport, #anns:1950
scene-0757, Arrive at busy intersection, bus, wait ... [18-08-30 19:25:08]   20s, boston-seaport, #anns:592
scene-0796, Scooter, peds on sidewalk, bus, cars, t... [18-10-02 02:52:24]   20s, singapore-queensto, #anns:708
scene-0916, Parki

In [2]:
class NuScenesTemporalDataset(Dataset):
    def __init__(self, nusc_env, scene_list, max_lidar_points=12000, nsweeps=3):
        self.nusc = nusc_env
        self.samples = []
        self.max_lidar_points = max_lidar_points
        self.nsweeps = nsweeps # Number of frames/sweeps to fuse
        
        for scene in scene_list:
            current_token = scene['first_sample_token']
            while current_token != '':
                sample = self.nusc.get('sample', current_token)
                self.samples.append(sample)
                current_token = sample['next']
                
        self.img_transform = transforms.Compose([
            transforms.Resize((224, 400)), 
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        print(f"Pre-loading HD Maps... (Configured for {nsweeps}-sweep temporal fusion)")
        locations = ['boston-seaport', 'singapore-onenorth', 'singapore-hollandvillage', 'singapore-queenstown']
        self.nusc_maps = {loc: NuScenesMap(dataroot=self.nusc.dataroot, map_name=loc) for loc in locations}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # --- 1. Camera Image (Current Frame Only to save VRAM) ---
        cam_token = sample['data']['CAM_FRONT']
        cam_data = self.nusc.get('sample_data', cam_token)
        image = Image.open(self.nusc.get_sample_data_path(cam_token)).convert('RGB')
        img_tensor = self.img_transform(image)
        
        # --- 2. Temporal RADAR FUSION ---
        # from_file_multisweep automatically aligns past sweeps to the current sensor position
        radar_pc, radar_times = RadarPointCloud.from_file_multisweep(
            self.nusc, sample, chan='RADAR_FRONT', ref_chan='RADAR_FRONT', nsweeps=self.nsweeps
        )
        
        # Extract x, y, z, vx_comp, vy_comp (indices 0, 1, 2, 8, 9)
        radar_base_features = radar_pc.points[[0, 1, 2, 8, 9], :]
        
        # Stack the time lag array to create a 6th feature channel (Delta t)
        radar_temporal = np.vstack([radar_base_features, radar_times]).T
        radar_tensor = torch.tensor(radar_temporal, dtype=torch.float32)
        
        # --- 3. Temporal LiDAR FUSION ---
        lidar_pc, lidar_times = LidarPointCloud.from_file_multisweep(
            self.nusc, sample, chan='LIDAR_TOP', ref_chan='LIDAR_TOP', nsweeps=self.nsweeps
        )
        
        # Extract x, y, z, intensity
        lidar_base_features = lidar_pc.points[:4, :]
        
        # Stack time lag to create a 5th feature channel
        lidar_temporal = np.vstack([lidar_base_features, lidar_times]).T
        
        # VRAM Safeguard Downsampling
        num_points = lidar_temporal.shape[0]
        if num_points >= self.max_lidar_points:
            indices = np.random.choice(num_points, self.max_lidar_points, replace=False)
        else:
            indices = np.random.choice(num_points, self.max_lidar_points, replace=True)
            
        lidar_tensor = torch.tensor(lidar_temporal[indices, :], dtype=torch.float32)
        
        # --- 4. Semantic BEV Map (Ground Truth) ---
        scene = self.nusc.get('scene', sample['scene_token'])
        location = self.nusc.get('log', scene['log_token'])['location']
        ego_pose = self.nusc.get('ego_pose', cam_data['ego_pose_token'])
        
        patch_box = (ego_pose['translation'][0], ego_pose['translation'][1], 40, 40)
        patch_angle = Quaternion(ego_pose['rotation']).yaw_pitch_roll[0]
        
        mask = self.nusc_maps[location].get_map_mask(
            patch_box=patch_box, patch_angle=patch_angle, layer_names=['drivable_area'], canvas_size=(200, 200)
        )
        mask_tensor = torch.tensor(mask[0], dtype=torch.float32).unsqueeze(0)
        
        return {
            'image': img_tensor,
            'radar': radar_tensor, # Now has 6 features!
            'lidar': lidar_tensor, # Now has 5 features!
            'label_mask': mask_tensor
        }

In [9]:
import torch
import torch.nn as nn
import torchvision.models as models

class PointNetExtractor(nn.Module):
    """Extracts features from an unordered point cloud (N points, C channels)"""
    def __init__(self, in_channels, out_features):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, out_features, kernel_size=1),
            nn.BatchNorm1d(out_features),
            nn.ReLU()
        )
        
    def forward(self, x):
        x = x.transpose(1, 2) 
        x = self.mlp(x)
        x = torch.max(x, dim=2, keepdim=False)[0] 
        return x

class BEVFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 1. Image Backbone 
        mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.image_backbone = nn.Sequential(*list(mobilenet.features.children()))
        self.image_pool = nn.AdaptiveAvgPool2d((1, 1)) 
        
        # 2. Temporal Point Cloud Backbones
        # UPDATED: LiDAR now expects 5 channels (x, y, z, intensity, time)
        self.lidar_net = PointNetExtractor(in_channels=5, out_features=128)
        
        # UPDATED: RADAR now expects 6 channels (x, y, z, vx, vy, time)
        self.radar_net = PointNetExtractor(in_channels=6, out_features=64)
        
        # 3. Fusion Layer
        self.fc = nn.Linear(1472, 256 * 5 * 5)
        
        # 4. BEV Decoder 
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),   
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),    
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=5, stride=5, padding=0),    
            nn.Conv2d(16, 1, kernel_size=3, padding=1)
        )

    def forward(self, image, lidar, radar):
        img_feat = self.image_pool(self.image_backbone(image)).flatten(1)
        lidar_feat = self.lidar_net(lidar)
        radar_feat = self.radar_net(radar)
        fused = torch.cat([img_feat, lidar_feat, radar_feat], dim=1)
        x = self.fc(fused).view(-1, 256, 5, 5)
        out = self.decoder(x)
        return out

In [12]:
# 1. Initialize the temporal dataset (3 sweeps)
temporal_dataset = NuScenesTemporalDataset(nusc, nusc.scene[:2], nsweeps=3)
temporal_loader = DataLoader(temporal_dataset, batch_size=1, shuffle=True, num_workers=0)

# 2. Grab a batch
batch = next(iter(temporal_loader))

# 3. Initialize the updated model
model = BEVFusionModel()

# 4. Pass the temporal data through
predictions = model(batch['image'], batch['lidar'], batch['radar'])

print("✅ Temporal Forward Pass Successful!")
print("-" * 35)
print(f"LiDAR Input Shape: {batch['lidar'].shape} -> [Batch, Points, 5 Features]")
print(f"RADAR Input Shape: {batch['radar'].shape} -> [Batch, Points, 6 Features]")
print(f"Output Prediction Shape: {predictions.shape}")

Pre-loading HD Maps... (Configured for 3-sweep temporal fusion)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /home/akashpq/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████████████████████████████████████████████████████████████████████████| 13.6M/13.6M [00:05<00:00, 2.72MB/s]


✅ Temporal Forward Pass Successful!
-----------------------------------
LiDAR Input Shape: torch.Size([1, 12000, 5]) -> [Batch, Points, 5 Features]
RADAR Input Shape: torch.Size([1, 181, 6]) -> [Batch, Points, 6 Features]
Output Prediction Shape: torch.Size([1, 1, 200, 200])


In [18]:
import torch
import torch.optim as optim
from torch.amp import autocast, GradScaler
import os

# 1. Hardware Initialization & Memory Flush
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache() # Clear out any lingering test tensors
torch.backends.cudnn.benchmark = True 

# 2. Initialize the Temporal Model and Optimizer
model = BEVFusionModel().to(device)
criterion = torch.nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scaler = GradScaler('cuda')

# 3. Initialize the Full Temporal DataLoader
# Using 10 scenes to give the network a solid variety of intersections and traffic
print("Loading Temporal Dataset...")
temporal_dataset = NuScenesTemporalDataset(nusc, nusc.scene[:10], nsweeps=3)
train_loader = DataLoader(temporal_dataset, batch_size=1, shuffle=True, num_workers=0)

# 4. Training Configuration
epochs = 5 
save_dir = "./temporal_saved_models"
os.makedirs(save_dir, exist_ok=True)

print(f"🚀 Starting Temporal Training on {device}...")
print("-" * 45)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for batch_idx, batch in enumerate(train_loader):
        # A. Move temporal tensors to GPU
        images = batch['image'].to(device)
        lidar = batch['lidar'].to(device)
        radar = batch['radar'].to(device)
        targets = batch['label_mask'].to(device)
        
        optimizer.zero_grad()
        
        # B. Mixed Precision Forward & Backward Pass
        with autocast('cuda'):
            predictions = model(images, lidar, radar)
            loss = criterion(predictions, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
        # C. Print progress every 20 batches
        if (batch_idx + 1) % 20 == 0:
            avg_loss = running_loss / 20
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx+1}/{len(train_loader)}] | Avg BCE Loss: {avg_loss:.4f}")
            running_loss = 0.0 
            
    # 5. Save Model Checkpoint
    checkpoint_path = os.path.join(save_dir, f"temporal_bev_epoch_{epoch+1}.pth")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"💾 Epoch {epoch+1} complete. Model saved to: {checkpoint_path}")
    print("-" * 45)

print("✅ Temporal Training Complete!")

Loading Temporal Dataset...
Pre-loading HD Maps... (Configured for 3-sweep temporal fusion)
🚀 Starting Temporal Training on cuda...
---------------------------------------------
Epoch [1/5] | Batch [20/404] | Avg BCE Loss: 0.6915
Epoch [1/5] | Batch [40/404] | Avg BCE Loss: 0.6889
Epoch [1/5] | Batch [60/404] | Avg BCE Loss: 0.6710
Epoch [1/5] | Batch [80/404] | Avg BCE Loss: 0.6518
Epoch [1/5] | Batch [100/404] | Avg BCE Loss: 0.6536
Epoch [1/5] | Batch [120/404] | Avg BCE Loss: 0.6395
Epoch [1/5] | Batch [140/404] | Avg BCE Loss: 0.6281
Epoch [1/5] | Batch [160/404] | Avg BCE Loss: 0.6440
Epoch [1/5] | Batch [180/404] | Avg BCE Loss: 0.6322
Epoch [1/5] | Batch [200/404] | Avg BCE Loss: 0.6357
Epoch [1/5] | Batch [220/404] | Avg BCE Loss: 0.6196
Epoch [1/5] | Batch [240/404] | Avg BCE Loss: 0.6072
Epoch [1/5] | Batch [260/404] | Avg BCE Loss: 0.6109
Epoch [1/5] | Batch [280/404] | Avg BCE Loss: 0.6021
Epoch [1/5] | Batch [300/404] | Avg BCE Loss: 0.6462
Epoch [1/5] | Batch [320/404] |

In [19]:
import torch
import numpy as np

# 1. Hardware and Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BEVFusionModel().to(device)

# 2. Load the trained weights
best_model_path = "./temporal_saved_models/temporal_bev_epoch_5.pth"
model.load_state_dict(torch.load(best_model_path))
model.eval() # Set to evaluation mode (critical for BatchNorm layers)
print(f"✅ Loaded weights from: {best_model_path}")

# 3. Create a Validation DataLoader (Using Unseen Scenes 10 to 12)
val_dataset = NuScenesTemporalDataset(nusc, nusc.scene[8:10], nsweeps=3)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

# 4. IoU Metric Function
def calculate_iou(pred_mask, true_mask, threshold=0.5):
    """Calculates the Intersection over Union for binary masks."""
    # Binarize the prediction based on the confidence threshold
    pred_bin = (pred_mask > threshold).astype(bool)
    true_bin = true_mask.astype(bool)
    
    intersection = np.logical_and(pred_bin, true_bin).sum()
    union = np.logical_or(pred_bin, true_bin).sum()
    
    if union == 0:
        return 0.0 # Avoid division by zero if both masks are empty
    return intersection / union

# 5. Evaluation Loop
total_iou = 0.0
num_batches = len(val_loader)

print(f"Starting Evaluation on {num_batches} unseen batches...")
print("-" * 45)

with torch.no_grad(): # Disable gradients for faster, memory-safe inference
    for batch_idx, batch in enumerate(val_loader):
        # Move inputs to GPU
        images = batch['image'].to(device)
        lidar = batch['lidar'].to(device)
        radar = batch['radar'].to(device)
        targets = batch['label_mask']
        
        # Forward pass with Mixed Precision
        with torch.amp.autocast('cuda'):
            raw_logits = model(images, lidar, radar)
            probabilities = torch.sigmoid(raw_logits)
            
        # Move prediction to CPU and numpy for metric calculation
        pred_numpy = probabilities.squeeze().cpu().numpy()
        target_numpy = targets.squeeze().cpu().numpy()
        
        # Calculate IoU
        batch_iou = calculate_iou(pred_numpy, target_numpy)
        total_iou += batch_iou
        
        if (batch_idx + 1) % 10 == 0:
            print(f"Evaluated Batch [{batch_idx+1}/{num_batches}] | Current Batch IoU: {batch_iou:.4f}")

# 6. Final Results
mean_iou = total_iou / num_batches
print("-" * 45)
print(f"🏆 Final Mean IoU on Unseen Data: {mean_iou * 100:.2f}%")

✅ Loaded weights from: ./temporal_saved_models/temporal_bev_epoch_5.pth
Pre-loading HD Maps... (Configured for 3-sweep temporal fusion)
Starting Evaluation on 80 unseen batches...
---------------------------------------------
Evaluated Batch [10/80] | Current Batch IoU: 0.4622
Evaluated Batch [20/80] | Current Batch IoU: 0.4592
Evaluated Batch [30/80] | Current Batch IoU: 0.2991
Evaluated Batch [40/80] | Current Batch IoU: 0.2574
Evaluated Batch [50/80] | Current Batch IoU: 0.8249
Evaluated Batch [60/80] | Current Batch IoU: 0.7362
Evaluated Batch [70/80] | Current Batch IoU: 0.7009
Evaluated Batch [80/80] | Current Batch IoU: 0.7831
---------------------------------------------
🏆 Final Mean IoU on Unseen Data: 56.41%
